# Pascal's Triangle and Coin Flips

*Math Concepts for Developers final project*

**Author:** Nina Georgieva

This project uses only the Python standard library, so it runs anywhere with no extra packages to install.

## 1. Problem formulation

When I flip a number of fair coins many times, how often should I expect to see 0 heads, 1 head, 2 heads, and so on?

There is a famous number pattern called **Pascal's triangle** that is supposed to predict these chances. In this project I check whether it actually works: I use the triangle to predict the probabilities, then I flip coins thousands of times in a simulation and compare the real results to the prediction.

**Why it matters:** this is a simple, hands-on way to see the link between a pure-math pattern and real random events. The same idea (the binomial distribution) shows up all the time in statistics, data science, and machine learning when we count how often something with two outcomes happens.

**Assumption:** every coin is fair, meaning heads and tails each have a probability of 0.5, and the flips do not affect each other.


## 2. Previous work and similar approaches

Connecting Pascal's triangle, the binomial distribution, and coin flips is a well-known idea, and many people have written about it before. A few examples:

- **Coin toss and the binomial theorem in Python** — a short script that links coin tosses to binomial probabilities: https://gist.github.com/jrjames83/2b922d36e81a9057afe71ea21dba86cb
- **experiment_coin_toss** — simulates very large numbers of coin tosses and watches the proportion of heads settle near 50%: https://github.com/JIVJGFJZKF/experiment_coin_toss


## 3. The math (kept simple)

Pascal's triangle is built row by row. Each row starts and ends with 1, and every number in the middle is the sum of the two numbers directly above it:

```
          1
        1   1
      1   2   1
    1   3   3   1
  1   4   6   4   1
```
How it connects to coins:

- Row `n` (counting from row 0 at the top) describes flipping `n` coins.
- The numbers in the row count how many ways you can get 0, 1, 2, ... heads.
- The row total is `2^n`, which is the number of all equally likely flip results.
- Dividing each number by the total gives the probability of that many heads.

**Example (3 coins):** row 3 is `1 3 3 1`, and the total is `1+3+3+1 = 8`. So:

- P(0 heads) = 1/8
- P(1 head) = 3/8
- P(2 heads) = 3/8
- P(3 heads) = 1/8

These triangle numbers are exactly the **binomial coefficients** $\binom{n}{k} = \dfrac{n!}{k!\,(n-k)!}$ from the course formula sheet. Building the triangle row by row is just an easy way to get them without computing factorials.

**Why do we add the two numbers above?** Think about adding one more coin to a smaller case. To get `k` heads in `n` coins, that last coin is either tails(so you already needed `k` heads in the first `n-1` coins) or heads (so you needed `k-1` heads in the first `n-1` coins). Adding those two possibilities together is exactly the rule "each number is the sum of the two above it."
That is why Pascal's triangle counts coin-flip outcomes correctly.

**Reading the 3-coin example by listing outcomes:** with 3 coins there are `2^3 = 8` equally likely results. Writing them all out (H = heads, T = tails): TTT; HTT, THT, TTH; HHT, HTH, THH; HHH. Grouping by number of heads gives 1, 3, 3, 1 — the same as row 3 of the triangle. This is a small check that the pattern and the real outcomes agree before we test it with a bigger simulation.

## 4. Building Pascal's triangle

The helper functions live in `pascal.py`. First we build and print the triangle.



In [1]:
import random
from pascal import (
    build_pascal_triangle,
    row_probabilities,
    flip_coins,
    counts_to_frequencies,
    text_bar_chart,
)

# Fix the random seed so the simulation gives the same numbers every run.
random.seed(42)

triangle = build_pascal_triangle(6)
for row in triangle:
    print(row)

[1]
[1, 1]
[1, 2, 1]
[1, 3, 3, 1]
[1, 4, 6, 4, 1]
[1, 5, 10, 10, 5, 1]


## 5. Simulating coin flips

We pick 5 coins and flip them 10,000 times. We compare the probabilities predicted by the triangle against the frequencies we actually get from the simulation.

In [8]:
n_coins = 5
n_trials = 10000

# Prediction from Pascal's triangle (row 5)
predicted = row_probabilities(build_pascal_triangle(n_coins + 1)[n_coins])

# Simulation
counts = flip_coins(n_coins, n_trials)
simulated = counts_to_frequencies(counts, n_trials)

print('heads | predicted | simulated')
for k in range(n_coins + 1):
    print(f'{k:5d} | {predicted[k]:9.4f} | {simulated[k]:9.4f}')

heads | predicted | simulated
    0 |    0.0312 |    0.0311
    1 |    0.1562 |    0.1579
    2 |    0.3125 |    0.3132
    3 |    0.3125 |    0.3146
    4 |    0.1562 |    0.1521
    5 |    0.0312 |    0.0311


## 6. Experiment: does more flipping give a better match?

If the prediction is correct, then flipping more times should make the simulated frequencies get closer to the predicted probabilities. We measure the \"total error\" as the sum of the differences across all outcomes, and we try several trial sizes.

In [7]:
for trials in [100, 1000, 10000, 100000]:
    sim = counts_to_frequencies(flip_coins(n_coins, trials), trials)
    total_error = sum(abs(sim[k] - predicted[k]) for k in range(n_coins + 1))
    print(f'trials={trials:7d}  total error={total_error:.4f}')

trials=    100  total error=0.1300
trials=   1000  total error=0.1070
trials=  10000  total error=0.0168
trials= 100000  total error=0.0037


In [ ]:
The error clearly gets smaller as the number of trials grows. With only 100 flips the match is rough, but with 100,000 flips it is very close. This is exactly what we expect: more data makes the simulation settle onto the true probabilities.

## 7. Chart: predicted vs. simulated

A simple text bar chart (no extra libraries) makes the comparison easy to see. The top bar of each pair is the triangle's prediction, the bottom bar is the simulation.


In [9]:
text_bar_chart(predicted, simulated)

heads | predicted                    | simulated
----------------------------------------------------------------------
    0 | ##                           | ##
      | 0.0312                       | 0.0311
    1 | ########                     | ########
      | 0.1562                       | 0.1579
    2 | ################             | ################
      | 0.3125                       | 0.3132
    3 | ################             | ################
      | 0.3125                       | 0.3146
    4 | ########                     | ########
      | 0.1562                       | 0.1521
    5 | ##                           | ##
      | 0.0312                       | 0.0311


## 8. Conclusion

Pascal's triangle does predict the coin-flip probabilities. When I flipped 5 coins many times, the simulated frequencies matched the probabilities taken from row 5 of the triangle. The experiment in section 5 showed that the more times I flipped, the smaller the difference between prediction and simulation became \u2014 the total error dropped from about 0.12 at 100 trials to about 0.009 at 100,000 trials.

So a pure number pattern (built only with additions) correctly describes a real random process. This is the binomial distribution in action.

**Possible next steps:** try an unfair coin (for example a 0.7 chance of heads) and check how the prediction would need to change, or try a larger number of coins and watch the familiar bell shape appear.

## References

- Wikipedia, \"Pascal's triangle\": https://en.wikipedia.org/wiki/Pascal%27s_triangle
- Wikipedia, \"Binomial distribution\": https://en.wikipedia.org/wiki/Binomial_distribution
- Course materials, Math Concepts for Developers
- Python standard library documentation, `random` module: https://docs.python.org/3/library/random.html
